# Quantum Optimization for Distributed Order Management
### WISER × Nestlé Global Quantum+AI Program 2026 — Team White Collars

This notebook runs the full pipeline end to end on the real Nestlé DOM data pack:
1. Load & document real data, extract a tractable focus-order subset
2. Classical baselines (default-assignment, greedy) + exact ILP
3. Quantum-inspired algorithm comparison (5 methods) on the real subset
4. Real gate-based QAOA circuit on a small subset, vs. brute-force optimum

**Setup:** place the `DOM-data` folder (from the challenge workspace) in the
same directory as this notebook, or set the `DOM_DATA_DIR` environment
variable to its path.


In [1]:
# Install dependencies (skip if already installed)
# !pip install pulp dimod dwave-samplers qiskit qiskit-aer pandas numpy scipy --quiet


## 1. Data loading — real Nestlé focus orders

In [2]:
"""
Loads the real Nestle DOM data pack and extracts a tractable subset of
FOCUS ORDERS (IsInvAvail == 'N', per the criteria in DOM Equations.docx)
into the same instance schema used by baseline_pulp.py / qubo_qaoa.py.

For tractability we take single-SKU focus orders only (359 focus orders
exist in total; most are single-SKU). Multi-SKU orders need the fuller
Cases_osdp formulation from the Equations doc -- noted as a scaling
extension in Task 5.
"""
import pandas as pd

import os
DATA_DIR = os.environ.get("DOM_DATA_DIR", "./DOM-data")


def load_real_instance(n_focus_orders=25, seed=7):
    """n_focus_orders: pass None (or a number >= 93) to use ALL single-SKU focus orders."""
    orders = pd.read_csv(f"{DATA_DIR}/input data/input_order data.csv")
    ship = pd.read_csv(f"{DATA_DIR}/input data/input_shipping_cost_data.csv")
    cap = pd.read_csv(f"{DATA_DIR}/input data/input_capacity_planning.csv")
    dock = pd.read_csv(f"{DATA_DIR}/input data/input_dock_capacity.csv")
    tput = pd.read_csv(f"{DATA_DIR}/input data/input_throughput_capacity.csv")

    # --- pick single-SKU focus orders (IsInvAvail == 'N') ---
    focus_lines = orders[orders["IsInvAvail"] == "N"].copy()
    line_counts = focus_lines.groupby("LoadNumber").size()
    single_sku_orders = line_counts[line_counts == 1].index
    focus_lines = focus_lines[focus_lines["LoadNumber"].isin(single_sku_orders)]

    picked = focus_lines.sort_values("Order_SKU_Revenue", ascending=False)
    if n_focus_orders is not None:
        picked = picked.head(n_focus_orders)

    order_ids = picked["LoadNumber"].tolist()
    order_sku = dict(zip(picked["LoadNumber"], picked["MaterialNumber"]))
    default_dc = dict(zip(picked["LoadNumber"], picked["Plant"]))
    value = dict(zip(picked["LoadNumber"], picked["Order_SKU_Revenue"]))
    order_zip = dict(zip(picked["LoadNumber"], picked["ZipCode"]))
    order_qty = dict(zip(picked["LoadNumber"], picked["OrderedQty_converted"]))
    order_rdd = dict(zip(picked["LoadNumber"], picked["RequestedDeliveryDate"]))
    # penalty: use FixedPenalty if present/nonzero, else a fraction of revenue via Penaltyforpotentialcuts
    penalty = {}
    for _, row in picked.iterrows():
        fp = row.get("FixedPenalty", 0)
        pct = row.get("Penaltyforpotentialcuts", 0)
        fp = float(fp) if pd.notna(fp) else 0.0
        pct = float(pct) if pd.notna(pct) else 0.0
        if fp <= 0 and pct <= 0:
            pct = 0.03  # dataset-wide median Penaltyforpotentialcuts, used when the field is missing
        penalty[row["LoadNumber"]] = round(fp if fp > 0 else pct * row["Order_SKU_Revenue"], 2)

    dcs = sorted(set(default_dc.values()) | set(ship["Plant"].unique()) & set(orders["Plant"].unique()))
    # restrict candidate DCs to those that actually appear as a default DC anywhere (keeps model realistic/tractable)
    dcs = sorted(orders["Plant"].unique().tolist())
    skus = sorted(set(order_sku.values()))

    # --- shipping cost: order -> DC, via ZipCode == TargetZip ---
    ship_cost = {}
    ship_lookup = ship.set_index(["TargetZip", "Plant"])["Shipping_Cost"].to_dict()
    for o in order_ids:
        z = order_zip[o]
        for d in dcs:
            c = ship_lookup.get((z, d))
            if c is None:
                # no direct lane priced for this DC/zip -> treat as a distant/unpriced lane
                c = ship.loc[ship["Plant"] == d, "Shipping_Cost"].mean() * 1.5
            ship_cost[(o, d)] = round(float(c), 2)

    # --- inventory: (DC, SKU) capacity from capacity_planning on the order's specific RDD ---
    cap["DATE"] = pd.to_datetime(cap["DATE"])
    cap_idx = cap.set_index(["LocationID", "MaterialID", "DATE"])["Available_inventory"]
    inventory = {}
    for d in dcs:
        for s in skus:
            rdd_dates = [pd.to_datetime(order_rdd[o]) for o in order_ids
                         if order_sku[o] == s and default_dc[o] == d]
            avail = None
            if rdd_dates:
                try:
                    avail = cap_idx.loc[(d, s, rdd_dates[0])]
                except KeyError:
                    avail = None
            if avail is None:
                rows = cap[(cap["LocationID"] == d) & (cap["MaterialID"] == s)]
                avail = rows.sort_values("DATE")["Available_inventory"].iloc[-1] if len(rows) else 0
            inventory[(d, s)] = max(0, int(round(avail))) if pd.notna(avail) else 0

    # --- throughput: use mean daily order_count capacity proxy per DC (90th pct of util as a soft cap) ---
    throughput = {}
    for d in dcs:
        rows = tput[tput["Plant"] == d]
        throughput[d] = int(rows["order_count"].quantile(0.75)) if len(rows) else 5
        throughput[d] = max(throughput[d], len(order_ids))  # don't let it be trivially binding at this small scale

    return {
        "orders": order_ids, "dcs": dcs, "skus": skus,
        "order_sku": order_sku, "default_dc": default_dc,
        "value": value, "ship_cost": ship_cost, "penalty": penalty,
        "inventory": inventory, "throughput": throughput,
        "_meta": {"order_zip": order_zip, "order_qty": order_qty, "order_rdd": order_rdd},
    }


if __name__ == "__main__":
    inst = load_real_instance(n_focus_orders=None)
    print(f"Orders: {len(inst['orders'])}, DCs: {len(inst['dcs'])}, SKUs: {len(inst['skus'])}")
    print(f"DCs: {inst['dcs']}")
    print()
    for o in inst["orders"][:5]:
        d = inst["default_dc"][o]
        s = inst["order_sku"][o]
        print(f"{o}: default_dc={d} sku={s} value={inst['value'][o]:.2f} "
              f"penalty={inst['penalty'][o]:.2f} ship_to_default={inst['ship_cost'][(o,d)]:.2f} "
              f"inv_at_default={inst['inventory'][(d,s)]}")


Orders: 93, DCs: 8, SKUs: 58
DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773]

U108967142: default_dc=5420 sku=12470852 value=198720.00 penalty=5961.60 ship_to_default=503.00 inv_at_default=606
U600106015: default_dc=5083 sku=12154829 value=85136.00 penalty=2554.08 ship_to_default=1071.00 inv_at_default=0
U108966275: default_dc=5620 sku=12549725 value=65673.00 penalty=1970.19 ship_to_default=335.00 inv_at_default=0
U108955078: default_dc=5410 sku=12484724 value=21310.00 penalty=639.30 ship_to_default=323.00 inv_at_default=0
U108963192: default_dc=5620 sku=12582086 value=18724.00 penalty=561.72 ship_to_default=5838.00 inv_at_default=0


In [3]:
inst = load_real_instance(n_focus_orders=None)
print(f"Loaded {len(inst['orders'])} focus orders, {len(inst['dcs'])} DCs, {len(inst['skus'])} SKUs")


Loaded 93 focus orders, 8 DCs, 58 SKUs


## 2. Classical baselines and exact ILP

In [4]:
"""
Task 4 (classical exact baseline): binary optimization model for DOM,
solved with PuLP/CBC.

Decision variable: x[o,d] = 1 if order o is assigned to DC d.

Objective (maximize):
    sum_{o,d} (value[o] - ship_cost[o,d]) * x[o,d]
    - sum_o penalty[o] * (1 - sum_d x[o,d])

Constraints:
    - each order assigned to at most one DC
    - per-DC, per-SKU inventory limit
    - per-DC throughput (order count) cap
"""
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD, LpStatus


def solve(inst):
    orders, dcs, skus = inst["orders"], inst["dcs"], inst["skus"]
    value, ship_cost, penalty = inst["value"], inst["ship_cost"], inst["penalty"]
    inventory, throughput, order_sku = inst["inventory"], inst["throughput"], inst["order_sku"]

    prob = LpProblem("DOM_baseline", LpMaximize)

    x = {(o, d): LpVariable(f"x_{o}_{d}", cat=LpBinary) for o in orders for d in dcs}

    # objective
    fulfil_term = lpSum((value[o] - ship_cost[(o, d)]) * x[(o, d)] for o in orders for d in dcs)
    penalty_term = lpSum(penalty[o] * (1 - lpSum(x[(o, d)] for d in dcs)) for o in orders)
    prob += fulfil_term - penalty_term

    # each order to at most one DC
    for o in orders:
        prob += lpSum(x[(o, d)] for d in dcs) <= 1

    # inventory limits per DC/SKU
    for d in dcs:
        for s in skus:
            orders_needing_s = [o for o in orders if order_sku[o] == s]
            prob += lpSum(x[(o, d)] for o in orders_needing_s) <= inventory[(d, s)]

    # throughput cap per DC
    for d in dcs:
        prob += lpSum(x[(o, d)] for o in orders) <= throughput[d]

    prob.solve(PULP_CBC_CMD(msg=False))

    assignment = {o: next((d for d in dcs if x[(o, d)].value() > 0.5), None) for o in orders}
    return prob, assignment


def report(inst, assignment, label):
    orders = inst["orders"]
    value, ship_cost, penalty = inst["value"], inst["ship_cost"], inst["penalty"]

    fulfilled = [o for o in orders if assignment[o] is not None]
    fill_rate = len(fulfilled) / len(orders)
    reassigned = [o for o in fulfilled if assignment[o] != inst["default_dc"][o]]
    total_shipping = sum(ship_cost[(o, assignment[o])] for o in fulfilled)
    total_value = sum(value[o] for o in fulfilled)
    total_penalty = sum(penalty[o] for o in orders if assignment[o] is None)
    objective = total_value - total_shipping - total_penalty

    print(f"\n--- {label} ---")
    print(f"Assignment: {assignment}")
    print(f"Objective value:     {objective:.2f}")
    print(f"Fill rate:           {fill_rate:.0%} ({len(fulfilled)}/{len(orders)})")
    print(f"Orders reassigned:   {len(reassigned)}  {reassigned}")
    print(f"Total shipping cost: {total_shipping:.2f}")
    print(f"Total penalty cost:  {total_penalty:.2f}")
    return {"objective": objective, "fill_rate": fill_rate, "reassigned": len(reassigned),
            "shipping": total_shipping, "penalty": total_penalty}


def default_assignment_baseline(inst):
    """Baseline 1: everyone stays at their default DC (feasibility not checked)."""
    orders, skus, dcs = inst["orders"], inst["skus"], inst["dcs"]
    inventory, order_sku, throughput = inst["inventory"], inst["order_sku"], inst["throughput"]
    remaining_inv = dict(inventory)
    remaining_cap = dict(throughput)
    assignment = {}
    for o in orders:
        d = inst["default_dc"][o]
        s = order_sku[o]
        if remaining_inv[(d, s)] > 0 and remaining_cap[d] > 0:
            assignment[o] = d
            remaining_inv[(d, s)] -= 1
            remaining_cap[d] -= 1
        else:
            assignment[o] = None
    return assignment


def greedy_baseline(inst):
    """Baseline 2: sort orders by value desc, assign to cheapest DC with capacity."""
    orders, dcs, skus = inst["orders"], inst["dcs"], inst["skus"]
    inventory, throughput, order_sku, ship_cost, value = (
        inst["inventory"], inst["throughput"], inst["order_sku"], inst["ship_cost"], inst["value"]
    )
    remaining_inv = dict(inventory)
    remaining_cap = dict(throughput)
    assignment = {}
    for o in sorted(orders, key=lambda o: -value[o]):
        s = order_sku[o]
        candidates = [d for d in dcs if remaining_inv[(d, s)] > 0 and remaining_cap[d] > 0]
        if not candidates:
            assignment[o] = None
            continue
        best_d = min(candidates, key=lambda d: ship_cost[(o, d)])
        assignment[o] = best_d
        remaining_inv[(best_d, s)] -= 1
        remaining_cap[best_d] -= 1
    return assignment




In [5]:
a1 = default_assignment_baseline(inst)
m1 = report(inst, a1, "Baseline 1: Default-assignment")

a2 = greedy_baseline(inst)
m2 = report(inst, a2, "Baseline 2: Greedy (value-desc, cheapest feasible DC)")

prob, a3 = solve(inst)
from pulp import LpStatus
print(f"\nILP solver status: {LpStatus[prob.status]}")
m3 = report(inst, a3, "ILP baseline (PuLP/CBC, optimal)")

print("\n=== Summary ===")
print(f"{'Method':35s} {'Objective':>10s} {'Fill':>6s} {'Reassigned':>11s} {'Ship':>8s} {'Penalty':>8s}")
for label, m in [("Default-assignment", m1), ("Greedy heuristic", m2), ("ILP (optimal)", m3)]:
    print(f"{label:35s} {m['objective']:10.2f} {m['fill_rate']:6.0%} {m['reassigned']:11d} {m['shipping']:8.2f} {m['penalty']:8.2f}")



--- Baseline 1: Default-assignment ---
Assignment: {'U108967142': 5420, 'U600106015': None, 'U108966275': None, 'U108955078': None, 'U108963192': None, 'U108971149': None, 'U600105961': None, 'U108948387': None, 'U108963193': None, 'U108963875': None, 'U108969732': 5620, 'U108964092': 5420, 'U108972115': 5410, 'U600105965': None, 'U108969029': None, 'U108969030': None, 'U108969031': None, 'U108969474': None, 'U108971488': None, 'U108968443': None, 'U108968445': None, 'U108972294': None, 'U108968446': None, 'U108971826': None, 'U108967211': None, 'U108971170': 5420, 'U108967674': 5410, 'U108968450': None, 'U108970018': None, 'U108965319': 5410, 'U108963187': None, 'U600106578': None, 'U108972784': None, 'U108970540': None, 'U108971805': None, 'U108971653': None, 'U108966974': 5620, 'U108963273': None, 'U108971660': None, 'U108971823': None, 'U108964185': None, 'U108971564': None, 'U108967711': 5490, 'U600106121': None, 'U108963573': None, 'U108968996': None, 'U600106172': None, 'U10896

## 3. QUBO encoding for quantum / quantum-inspired solving

In [6]:
"""
Task 3/4: QUBO encoding of the DOM problem for quantum / quantum-inspired
solving (QAOA-style), solved here with simulated annealing (a classical
quantum-inspired proxy — see notes at the bottom on the actual QAOA circuit).

--- Encoding ---
Binary variable per (order, DC) pair, same as the ILP: x[o,d] in {0,1}.
QAOA/QUBO solvers need an UNCONSTRAINED objective, so hard constraints
become quadratic PENALTY terms added to the cost function instead of
explicit inequality constraints:

  1. "at most one DC per order":  A * sum_o (sum_d x[o,d] - 1)^2   [only if forced to =1;
     since partial non-assignment is allowed, we instead penalize
     *multiple* assignments: A * sum_o sum_{d<d'} x[o,d]*x[o,d']]
  2. inventory capacity per (DC, SKU): B * sum_{d,s} ( sum_{o in s} x[o,d] - inv[d,s] )_+^2
     (approximated below as a quadratic over-capacity penalty)
  3. throughput cap per DC: C * (sum_o x[o,d] - throughput[d])_+^2  (approximated similarly)

Objective (to MINIMIZE, so we negate the ILP's maximization terms):
  H = -sum_{o,d}(value[o]-ship_cost[o,d]) x[o,d]
      + sum_o penalty[o] * (1 - sum_d x[o,d])       <- kept linear, already unconstrained-friendly
      + A * one-DC-violation terms
      + B * inventory-violation terms (soft)
      + C * throughput-violation terms (soft)

This gives a QUBO matrix Q such that H(x) = x^T Q x (+ constant), which is
exactly the form QAOA optimizes: encode Q as a cost Hamiltonian H_C over
one qubit per (order, DC) pair, alternate with a mixer Hamiltonian H_M
(sum of X_i), and optimize circuit angles (beta, gamma) to minimize <H_C>.
"""
import dimod
import neal


def build_qubo(inst, A=None, B=None, C=None):
    orders, dcs, skus = inst["orders"], inst["dcs"], inst["skus"]
    value, ship_cost, penalty = inst["value"], inst["ship_cost"], inst["penalty"]
    inventory, throughput, order_sku = inst["inventory"], inst["throughput"], inst["order_sku"]

    # scale constraint penalty weights relative to the objective's own magnitude
    # (fixed constants tuned for a small synthetic instance don't transfer to
    # real revenue figures in the hundreds of thousands)
    typical_term = max(value.values()) if value else 1
    if A is None:
        A = 2 * typical_term
    if B is None:
        B = 2 * typical_term
    if C is None:
        C = 2 * typical_term

    Q = {}  # dict[(var_i, var_j)] = coefficient, dimod's sparse QUBO format
    var = lambda o, d: f"x_{o}_{d}"

    def add(i, j, val):
        key = (i, j) if i <= j else (j, i)
        Q[key] = Q.get(key, 0) + val

    # --- linear reward/cost term: -(value - ship_cost) per (o,d), plus penalty bookkeeping ---
    for o in orders:
        for d in dcs:
            v = var(o, d)
            # negative because QUBO minimizes; we want to maximize (value-ship_cost) and
            # avoid the penalty (penalty[o] is incurred only when order stays unassigned,
            # i.e. contributes -penalty[o]*x[o,d] as a "reward" for assigning)
            add(v, v, -(value[o] - ship_cost[(o, d)]) - penalty[o])

    # --- constraint A: at most one DC per order -> penalize any pair d<d' both selected ---
    for o in orders:
        for i, d1 in enumerate(dcs):
            for d2 in dcs[i + 1:]:
                add(var(o, d1), var(o, d2), 2 * A)

    # --- constraint B: inventory capacity per (DC, SKU) ---
    # NOTE: a naive (sum x - cap)^2 penalty is symmetric -- it penalizes being
    # UNDER capacity just as much as OVER it, which wrongly discourages valid,
    # low-utilization assignments. We use a one-sided approximation instead:
    #   - if cap == 0: hard-exclude every assignment to that (DC,SKU) (large linear penalty)
    #   - if cap >= 1: only penalize pairwise co-selection beyond what cap allows
    #     (exact for cap==1 "at most one"; an approximation for cap>1 with group
    #     sizes above cap -- a known simplification, see Task 5 / feasibility-repair notes)
    for d in dcs:
        for s in skus:
            group = [o for o in orders if order_sku[o] == s]
            cap = inventory[(d, s)]
            if cap == 0:
                for o in group:
                    add(var(o, d), var(o, d), B)
            elif len(group) > cap:
                for i, o1 in enumerate(group):
                    for o2 in group[i + 1:]:
                        add(var(o1, d), var(o2, d), B)

    # --- constraint C: throughput cap per DC, same one-sided approach ---
    for d in dcs:
        cap = throughput[d]
        if cap == 0:
            for o in orders:
                add(var(o, d), var(o, d), C)
        elif len(orders) > cap:
            for i, o1 in enumerate(orders):
                for o2 in orders[i + 1:]:
                    add(var(o1, d), var(o2, d), C)

    return Q


def decode(sample, inst):
    orders, dcs = inst["orders"], inst["dcs"]
    assignment = {}
    for o in orders:
        chosen = [d for d in dcs if sample.get(f"x_{o}_{d}", 0) == 1]
        assignment[o] = chosen[0] if len(chosen) == 1 else None  # None if 0 or >1 (constraint violated)
    return assignment




## 4. Comparing multiple quantum-inspired algorithms at full scale (93 orders × 8 DCs)

In [7]:
"""
Task 4 (extended): compares MULTIPLE quantum / quantum-inspired algorithms
on the same QUBO, at the full 93-order real-data scale, plus a genuine
gate-based QAOA circuit run on a small subset (since 744 qubits is not
statevector-simulable). Reports which is the best-suited method for this
problem at this scale, against the ILP exact optimum as ground truth.

Algorithms compared on the full-scale QUBO (744 binary variables):
  1. Random sampling            - naive baseline, shows the QUBO isn't trivial
  2. Simulated Annealing (SA)   - classical thermal annealing
  3. Tabu Search                - classical local search with memory
  4. Steepest Descent           - greedy local search to a local optimum
  5. Path-Integral Quantum Monte Carlo (PIQMC) - simulates quantum
     annealing's tunneling behaviour via Trotterized replicas; the
     closest classical proxy to how a real D-Wave quantum annealer
     would search this landscape

Then, on a SMALL subset (tractable for a real quantum circuit):
  6. QAOA (qiskit-aer statevector simulator) - actual gate-based quantum
     algorithm, compared against brute-force optimum on that subset
"""
import time
import numpy as np
import dimod
from dwave.samplers import (
    SimulatedAnnealingSampler, TabuSampler, SteepestDescentSampler,
    RandomSampler, PathIntegralAnnealingSampler,
)
report_metrics = report


def run_sampler(name, sampler, bqm, inst, **kwargs):
    t0 = time.time()
    result = sampler.sample(bqm, **kwargs)
    elapsed = time.time() - t0
    best_sample = result.first.sample
    assignment = decode(best_sample, inst)
    m = report_metrics(inst, assignment, name)
    m["label"] = name
    m["runtime_s"] = elapsed
    return m




In [8]:

print("=" * 70)
print("PART 1: Quantum-inspired algorithms at full scale (93 orders x 8 DCs)")
print("=" * 70)
inst = load_real_instance(n_focus_orders=None)
Q = build_qubo(inst)
bqm = dimod.BinaryQuadraticModel.from_qubo(Q)
print(f"QUBO size: {len(bqm.variables)} binary variables\n")

results = []
results.append(run_sampler("1. Random sampling", RandomSampler(), bqm, inst, num_reads=200))
results.append(run_sampler("2. Simulated Annealing", SimulatedAnnealingSampler(), bqm, inst, num_reads=200, seed=7))
results.append(run_sampler("3. Tabu Search", TabuSampler(), bqm, inst, num_reads=20, timeout=5000))
results.append(run_sampler("4. Steepest Descent", SteepestDescentSampler(), bqm, inst, num_reads=200, seed=7))
results.append(run_sampler("5. Path-Integral QMC", PathIntegralAnnealingSampler(), bqm, inst, num_reads=50, num_sweeps=200, seed=7))

print("\n=== Full-scale comparison (ILP optimum = 447,822 for reference) ===")
print(f"{'Method':28s} {'Objective':>12s} {'Fill':>6s} {'Reassigned':>11s} {'Runtime(s)':>11s}")
for m in results:
    print(f"{m['label']:28s} {m['objective']:12.2f} {m['fill_rate']:6.0%} {m['reassigned']:11d} {m['runtime_s']:11.3f}")

best = max(results, key=lambda m: m["objective"])
print(f"\nBest quantum-inspired method at this scale: {best['label']} "
      f"(objective {best['objective']:.2f}, {best['objective']/447822.39:.1%} of ILP optimum)")


PART 1: Quantum-inspired algorithms at full scale (93 orders x 8 DCs)


QUBO size: 744 binary variables


--- 1. Random sampling ---
Assignment: {'U108967142': None, 'U600106015': None, 'U108966275': 5773, 'U108955078': None, 'U108963192': None, 'U108971149': None, 'U600105961': None, 'U108948387': None, 'U108963193': None, 'U108963875': None, 'U108969732': None, 'U108964092': None, 'U108972115': None, 'U600105965': 5083, 'U108969029': None, 'U108969030': None, 'U108969031': None, 'U108969474': 5385, 'U108971488': None, 'U108968443': None, 'U108968445': None, 'U108972294': None, 'U108968446': None, 'U108971826': None, 'U108967211': None, 'U108971170': None, 'U108967674': None, 'U108968450': None, 'U108970018': None, 'U108965319': None, 'U108963187': None, 'U600106578': 5420, 'U108972784': 5385, 'U108970540': None, 'U108971805': None, 'U108971653': None, 'U108966974': None, 'U108963273': None, 'U108971660': 5385, 'U108971823': 5620, 'U108964185': None, 'U108971564': None, 'U108967711': None, 'U600106121': None, 'U108963573': None, 'U108968996': None, 'U6001


--- 2. Simulated Annealing ---
Assignment: {'U108967142': 5420, 'U600106015': None, 'U108966275': 5490, 'U108955078': 5385, 'U108963192': 5420, 'U108971149': 5410, 'U600105961': None, 'U108948387': 5385, 'U108963193': 5420, 'U108963875': None, 'U108969732': 5420, 'U108964092': 5420, 'U108972115': 5620, 'U600105965': None, 'U108969029': 5641, 'U108969030': 5641, 'U108969031': 5641, 'U108969474': None, 'U108971488': 5420, 'U108968443': 5420, 'U108968445': 5420, 'U108972294': 5420, 'U108968446': 5420, 'U108971826': 5641, 'U108967211': 5490, 'U108971170': 5420, 'U108967674': 5490, 'U108968450': 5620, 'U108970018': 5490, 'U108965319': 5490, 'U108963187': 5641, 'U600106578': None, 'U108972784': None, 'U108970540': 5490, 'U108971805': 5420, 'U108971653': 5410, 'U108966974': None, 'U108963273': None, 'U108971660': 5410, 'U108971823': None, 'U108964185': 5641, 'U108971564': 5083, 'U108967711': None, 'U600106121': None, 'U108963573': 5641, 'U108968996': None, 'U600106172': 5773, 'U108964093': 5


--- 3. Tabu Search ---
Assignment: {'U108967142': 5420, 'U600106015': None, 'U108966275': 5385, 'U108955078': 5385, 'U108963192': 5420, 'U108971149': 5641, 'U600105961': None, 'U108948387': 5385, 'U108963193': 5420, 'U108963875': 5420, 'U108969732': 5620, 'U108964092': 5420, 'U108972115': 5385, 'U600105965': None, 'U108969029': 5641, 'U108969030': 5641, 'U108969031': 5641, 'U108969474': 5420, 'U108971488': 5641, 'U108968443': 5420, 'U108968445': 5420, 'U108972294': 5420, 'U108968446': 5420, 'U108971826': 5641, 'U108967211': 5641, 'U108971170': 5420, 'U108967674': 5410, 'U108968450': 5385, 'U108970018': 5385, 'U108965319': 5410, 'U108963187': 5641, 'U600106578': 5773, 'U108972784': None, 'U108970540': 5490, 'U108971805': 5641, 'U108971653': 5420, 'U108966974': 5620, 'U108963273': None, 'U108971660': 5410, 'U108971823': None, 'U108964185': 5083, 'U108971564': 5490, 'U108967711': 5410, 'U600106121': None, 'U108963573': 5641, 'U108968996': None, 'U600106172': None, 'U108964093': 5641, 'U1


--- 5. Path-Integral QMC ---
Assignment: {'U108967142': 5420, 'U600106015': None, 'U108966275': 5385, 'U108955078': 5385, 'U108963192': 5420, 'U108971149': 5641, 'U600105961': None, 'U108948387': 5385, 'U108963193': None, 'U108963875': 5641, 'U108969732': 5620, 'U108964092': 5385, 'U108972115': 5490, 'U600105965': None, 'U108969029': 5641, 'U108969030': None, 'U108969031': None, 'U108969474': 5420, 'U108971488': 5641, 'U108968443': None, 'U108968445': 5420, 'U108972294': 5420, 'U108968446': 5420, 'U108971826': 5490, 'U108967211': 5641, 'U108971170': 5420, 'U108967674': 5420, 'U108968450': 5385, 'U108970018': None, 'U108965319': 5490, 'U108963187': 5420, 'U600106578': 5773, 'U108972784': None, 'U108970540': 5490, 'U108971805': 5385, 'U108971653': 5385, 'U108966974': 5620, 'U108963273': 5620, 'U108971660': 5410, 'U108971823': None, 'U108964185': 5385, 'U108971564': 5083, 'U108967711': 5641, 'U600106121': None, 'U108963573': 5490, 'U108968996': None, 'U600106172': None, 'U108964093': Non

## 5. Real gate-based QAOA circuit (small subset)

744 qubits is far beyond statevector-simulator feasibility (2^744 states),
so we demonstrate an actual QAOA circuit on a small 3-order × 2-DC (6-qubit)
subset, benchmarked against the brute-force optimum on that same subset.


In [9]:
"""
Task 4 (extended), Part 2: a hand-built QAOA circuit (qiskit-algorithms
0.4.0 is incompatible with the installed qiskit 2.x primitives interface,
so QAOA is implemented directly here -- standard textbook construction).
Run on a SMALL subset (6 qubits) where statevector simulation and
brute-force ground truth are both feasible, unlike the full 744-qubit
instance.
"""
import itertools
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit import QuantumCircuit
from qiskit_aer import AerSimulator
import dimod

report_metrics = report


def qubo_to_ising(bqm):
    """QUBO (x in {0,1}) -> Ising (z in {-1,+1}) via x = (1-z)/2."""
    variables = list(bqm.variables)
    idx = {v: i for i, v in enumerate(variables)}
    n = len(variables)
    h = np.zeros(n)
    J = {}
    offset = bqm.offset
    for v, bias in bqm.linear.items():
        i = idx[v]
        h[i] += -bias / 2
        offset += bias / 2
    for (u, v), bias in bqm.quadratic.items():
        i, j = idx[u], idx[v]
        J[(i, j)] = J.get((i, j), 0) + bias / 4
        h[i] += -bias / 4
        h[j] += -bias / 4
        offset += bias / 4
    return h, J, offset, variables


def qaoa_circuit(n, h, J, gammas, betas):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for gamma, beta in zip(gammas, betas):
        for i in range(n):
            if h[i] != 0:
                qc.rz(2 * gamma * h[i], i)
        for (i, j), coeff in J.items():
            if coeff != 0:
                qc.cx(i, j)
                qc.rz(2 * gamma * coeff, j)
                qc.cx(i, j)
        for i in range(n):
            qc.rx(2 * beta, i)
    qc.measure_all()
    return qc


def expected_cost(counts, h, J, offset):
    total_shots = sum(counts.values())
    exp = 0.0
    for bitstring, count in counts.items():
        z = np.array([1 - 2 * int(b) for b in bitstring[::-1]])  # qiskit bit order
        energy = offset + sum(h[i] * z[i] for i in range(len(h)))
        energy += sum(coeff * z[i] * z[j] for (i, j), coeff in J.items())
        exp += energy * count / total_shots
    return exp


def run_qaoa(h, J, offset, reps=2, shots=2048, seed=7):
    n = len(h)
    backend = AerSimulator(seed_simulator=seed)

    def objective(params):
        gammas, betas = params[:reps], params[reps:]
        qc = qaoa_circuit(n, h, J, gammas, betas)
        result = backend.run(qc, shots=shots).result()
        counts = result.get_counts()
        return expected_cost(counts, h, J, offset)

    x0 = np.random.default_rng(seed).uniform(0, np.pi, size=2 * reps)
    res = minimize(objective, x0, method="COBYLA", options={"maxiter": 150})

    gammas, betas = res.x[:reps], res.x[reps:]
    qc = qaoa_circuit(n, h, J, gammas, betas)
    counts = backend.run(qc, shots=shots).result().get_counts()
    best_bitstring = max(counts, key=counts.get)
    return best_bitstring, res.fun


def brute_force(bqm):
    variables = list(bqm.variables)
    best_energy, best_sample = None, None
    for bits in itertools.product([0, 1], repeat=len(variables)):
        sample = dict(zip(variables, bits))
        e = bqm.energy(sample)
        if best_energy is None or e < best_energy:
            best_energy, best_sample = e, sample
    return best_sample, best_energy




In [10]:

full_inst = load_real_instance(n_focus_orders=None)
orders_sub = full_inst["orders"][:3]
dcs_sub = full_inst["dcs"][:2]
skus_sub = sorted(set(full_inst["order_sku"][o] for o in orders_sub))

inst = {
    "orders": orders_sub, "dcs": dcs_sub, "skus": skus_sub,
    "order_sku": {o: full_inst["order_sku"][o] for o in orders_sub},
    "default_dc": {o: full_inst["default_dc"][o] for o in orders_sub},
    "value": {o: full_inst["value"][o] for o in orders_sub},
    "ship_cost": {(o, d): full_inst["ship_cost"][(o, d)] for o in orders_sub for d in dcs_sub},
    "penalty": {o: full_inst["penalty"][o] for o in orders_sub},
    "inventory": {(d, s): full_inst["inventory"][(d, s)] for d in dcs_sub for s in skus_sub},
    "throughput": {d: full_inst["throughput"][d] for d in dcs_sub},
}

Q = build_qubo(inst)
bqm = dimod.BinaryQuadraticModel.from_qubo(Q)
n = len(bqm.variables)
print(f"Small subset: {len(orders_sub)} orders x {len(dcs_sub)} DCs = {n} qubits\n")

bf_sample, bf_energy = brute_force(bqm)
bf_assignment = decode(bf_sample, inst)
bf_metrics = report_metrics(inst, bf_assignment, "Brute-force optimum (ground truth)")

h, J, offset, variables = qubo_to_ising(bqm)
best_bitstring, qaoa_energy_est = run_qaoa(h, J, offset, reps=2, shots=2048)

qaoa_sample = {variables[i]: int(bit) for i, bit in enumerate(best_bitstring[::-1])}
qaoa_assignment = decode(qaoa_sample, inst)
qaoa_metrics = report_metrics(inst, qaoa_assignment, "QAOA (hand-built circuit, p=2, qiskit-aer)")

print("\n=== Small-subset comparison (6-qubit real quantum circuit) ===")
print(f"{'Method':45s} {'Objective':>10s} {'Fill':>6s}")
print(f"{'Brute-force optimum':45s} {bf_metrics['objective']:10.2f} {bf_metrics['fill_rate']:6.0%}")
print(f"{'QAOA (real circuit, p=2, qiskit-aer)':45s} {qaoa_metrics['objective']:10.2f} {qaoa_metrics['fill_rate']:6.0%}")
gap = (bf_metrics['objective'] - qaoa_metrics['objective']) / max(abs(bf_metrics['objective']), 1)
print(f"\nQAOA optimality gap: {gap:.1%}")


Small subset: 3 orders x 2 DCs = 6 qubits


--- Brute-force optimum (ground truth) ---
Assignment: {'U108967142': 5385, 'U600106015': None, 'U108966275': 5385}
Objective value:     256411.92
Fill rate:           67% (2/3)
Orders reassigned:   2  ['U108967142', 'U108966275']
Total shipping cost: 5427.00
Total penalty cost:  2554.08



--- QAOA (hand-built circuit, p=2, qiskit-aer) ---
Assignment: {'U108967142': 5385, 'U600106015': None, 'U108966275': None}
Objective value:     191973.73
Fill rate:           33% (1/3)
Orders reassigned:   1  ['U108967142']
Total shipping cost: 2222.00
Total penalty cost:  4524.27

=== Small-subset comparison (6-qubit real quantum circuit) ===
Method                                         Objective   Fill
Brute-force optimum                            256411.92    67%
QAOA (real circuit, p=2, qiskit-aer)           191973.73    33%

QAOA optimality gap: 25.1%


## 6. Summary

- **ILP (exact)**: 447,822 objective — the reference optimum
- **Best quantum-inspired method**: Steepest Descent, 99.2% of optimum in 0.08s
- **Real QAOA (6 qubits)**: 25.1% optimality gap — illustrates current
  hardware/simulator scale limits for this problem
- **Recommendation**: use exact ILP or Steepest Descent in production at
  this scale; batch larger order sets (see Task 5 write-up) to keep future
  growth tractable.

See the accompanying technical report, data dictionary, and scaling
analysis for full discussion.
